# Phase 4.2: v2a-RSN Edgewise Localization

Compute edge-level instability metrics from v2a-RSN c-GC and c-GC* outputs.

**Goal**: Identify which edges drive global graph instability across conditioning depths.

**Output**:
- `edge_instability.csv`: Edge-level metrics (additions, deletions, instability frequency)
- `edge_fdr.csv`: FDR-corrected p-values (if bootstrap available)
- `deletion_heatmap.png`: Heatmap of edge deletions per depth
- `addition_heatmap.png`: Heatmap of edge additions per depth

In [ ]:
from __future__ import annotations

import sys
import json
import logging
import time
import pickle
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from markovianity_diagnostic.experiments.edge_localization import (
    compute_edge_instability_metrics
)

# Configuration
np.random.seed(42)
plt.style.use('default')
sns.set_palette("husl")

logger.info("✓ Imports successful")
print("✓ Imports successful")

In [ ]:
# Check for cached outputs
project_root = Path.cwd()
outputs_v2a = project_root / 'outputs' / 'v2a-RSNs'
output_dir = outputs_v2a / 'edge_localization'

expected_outputs = {
    'edge_instability.csv': output_dir / 'edge_instability.csv',
    'manifest.json': output_dir / 'manifest.json',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    logger.info("Outputs already exist - loading cached results")
    print("✅ Outputs already exist - loading cached results")
    print(f"Output directory: {output_dir}")
    for name, path in expected_outputs.items():
        size_mb = path.stat().st_size / (1024 * 1024)
        logger.info(f"  ✓ {name} ({size_mb:.2f} MB)")
        print(f"  ✓ {name} ({size_mb:.2f} MB)")
else:
    logger.info("No existing outputs - will run computation")
    print("⚠️ No existing outputs - will run computation")
    print(f"Output directory: {output_dir}")

## Load v2a-RSN c-GC and c-GC-star outputs

In [ ]:
if not outputs_exist:
    logger.info("Discovering v2a-RSN recordings...")
    start_discovery = time.time()
    
    # Define paths
    # Find all pickle files (adjacency dictionaries)
    cgc_pkl_files = list((outputs_v2a / 'c-GC').glob('*.pkl'))
    cgc_star_pkl_files = list((outputs_v2a / 'c-GC-star').glob('*.pkl'))

    logger.info(f"Found {len(cgc_pkl_files)} c-GC recordings")
    logger.info(f"Found {len(cgc_star_pkl_files)} c-GC-star recordings")
    print(f"Found {len(cgc_pkl_files)} c-GC recordings")
    print(f"Found {len(cgc_star_pkl_files)} c-GC-star recordings")

    # Extract recording IDs
    cgc_recordings = {f.stem: f for f in cgc_pkl_files}
    cgc_star_recordings = {f.stem: f for f in cgc_star_pkl_files}

    logger.info("\\nc-GC recordings (first 5):")
    print("\\nc-GC recordings:")
    for rid in sorted(cgc_recordings.keys())[:5]:
        logger.info(f"  {rid}")
        print(f"  {rid}")
        
    logger.info("\\nc-GC-star recordings (first 5):")
    print("\\nc-GC-star recordings:")
    for rid in sorted(cgc_star_recordings.keys())[:5]:
        logger.info(f"  {rid}")
        print(f"  {rid}")
    
    discovery_elapsed = time.time() - start_discovery
    logger.info(f"Discovery completed in {discovery_elapsed:.2f}s")
else:
    logger.info("Skipping file discovery (cached outputs)")
    print("Skipping file discovery (cached outputs)")

## Compute edge-level instability metrics for each recording and method

In [ ]:
if not outputs_exist:
    logger.info("Starting edge instability computation...")
    start_compute = time.time()
    
    def load_adjacency_dict(pkl_path):
        """Load adjacency dictionary from pickle file."""
        with open(pkl_path, 'rb') as f:
            adj_dict = pickle.load(f)
        return adj_dict

    # Process all recordings and methods
    all_edge_metrics = []

    for method_idx, (method, recordings) in enumerate([('c-GC', cgc_recordings), ('c-GC-star', cgc_star_recordings)], 1):
        logger.info(f"\n[{method_idx}/2] Processing {method}")
        print(f"\n{'='*60}")
        print(f"Processing {method}")
        print(f"{'='*60}")
        
        for rec_idx, (recording_id, pkl_path) in enumerate(sorted(recordings.items()), 1):
            logger.info(f"  [{rec_idx}/{len(recordings)}] {recording_id}...")
            try:
                # Load adjacency dictionary
                adj_dict = load_adjacency_dict(pkl_path)
                
                # Compute edge metrics
                edge_df = compute_edge_instability_metrics(
                    adj_dict=adj_dict,
                    recording=recording_id,
                    method=method,
                    bootstrap_adj_dict=None  # No bootstrap available from c-GC outputs
                )
                
                all_edge_metrics.append(edge_df)
                
                unstable_count = (edge_df['status'] == 'unstable').sum()
                stable_count = (edge_df['status'] == 'stable').sum()
                logger.info(f"    {recording_id:25s}: {len(edge_df):3d} edges, unstable: {unstable_count:3d}, stable: {stable_count:3d}")
                print(f"  {recording_id:25s}: {len(edge_df):3d} edges, unstable: {unstable_count:3d}, stable: {stable_count:3d}")
                
            except Exception as e:
                logger.error(f"    {recording_id:25s}: ERROR - {str(e)}")
                print(f"  {recording_id:25s}: ERROR - {str(e)}")

    # Combine all results
    edge_metrics_df = pd.concat(all_edge_metrics, ignore_index=True) if all_edge_metrics else pd.DataFrame()

    logger.info(f"\n{'='*60}")
    logger.info(f"Summary: {len(edge_metrics_df)} total edges across all recordings")
    logger.info(f"Unstable: {(edge_metrics_df['status'] == 'unstable').sum()}")
    logger.info(f"Stable: {(edge_metrics_df['status'] == 'stable').sum()}")
    logger.info(f"Novel: {(edge_metrics_df['status'] == 'novel').sum()}")
    logger.info(f"Lost: {(edge_metrics_df['status'] == 'lost').sum()}")
    
    compute_elapsed = time.time() - start_compute
    logger.info(f"Computation completed in {compute_elapsed:.2f}s")
    
    print(f"\n{'='*60}")
    print(f"Summary: {len(edge_metrics_df)} total edges")
else:
    logger.info("Skipping edge metric computation (cached outputs)")
    print("Skipping edge metric computation (cached outputs)")

In [ ]:
# Load cached edge metrics if they exist
if outputs_exist:
    edge_instability_csv = output_dir / 'edge_instability.csv'
    edge_metrics_df = pd.read_csv(edge_instability_csv)
    print(f"Loaded cached edge metrics: {len(edge_metrics_df)} edges")
    print(f"Status distribution:")
    print(edge_metrics_df['status'].value_counts() if 'status' in edge_metrics_df.columns else "N/A")

## Export edge metrics to CSV

In [ ]:
logger.info("Exporting edge metrics...")
start_export = time.time()

if not outputs_exist:
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Export edge instability metrics
    edge_instability_csv = output_dir / 'edge_instability.csv'
    edge_metrics_df.to_csv(edge_instability_csv, index=False)
    logger.info(f'Exported: {edge_instability_csv}')
    logger.info(f'  Shape: {edge_metrics_df.shape}')
    print(f'Exported: {edge_instability_csv}')
    print(f'  Shape: {edge_metrics_df.shape}')

    # Export FDR metrics (subset of columns for clarity)
    if 'q_value' in edge_metrics_df.columns:
        fdr_cols = ['recording', 'method', 'source', 'target', 
                    'instability_frequency', 'null_frequency', 'p_value', 'q_value']
        fdr_cols = [c for c in fdr_cols if c in edge_metrics_df.columns]
        edge_fdr_df = edge_metrics_df[fdr_cols].dropna(subset=['p_value'])
        
        edge_fdr_csv = output_dir / 'edge_fdr.csv'
        edge_fdr_df.to_csv(edge_fdr_csv, index=False)
        logger.info(f'Exported: {edge_fdr_csv}')
    
    export_elapsed = time.time() - start_export
    logger.info(f"Export completed in {export_elapsed:.2f}s")
else:
    edge_instability_csv = output_dir / 'edge_instability.csv'
    logger.info("Skipping export (loading from cache)")
    print("Skipping export (loading from cache)")

## Create edge deletion and addition heatmaps

In [ ]:
def create_edge_heatmaps(pkl_dict, recording_id, method, output_dir):
    """Create heatmaps for edge additions and deletions across depths."""
    depths = sorted(pkl_dict.keys())
    n_depths = len(depths)
    
    # Collect all edges
    all_edges = set()
    for adj_matrix in pkl_dict.values():
        rows, cols = np.where(adj_matrix == 1)
        for src, tgt in zip(rows, cols):
            if src != tgt:
                all_edges.add((src, tgt))
    
    if not all_edges:
        return None, None
    
    # Create matrices for deletions and additions
    deletion_matrix = np.zeros((len(all_edges), n_depths - 1))
    addition_matrix = np.zeros((len(all_edges), n_depths - 1))
    edge_labels = []
    
    for edge_idx, (src, tgt) in enumerate(sorted(all_edges)):
        edge_labels.append(f"{src}→{tgt}")
        
        # Track appearances across depths
        for i, depth in enumerate(depths[:-1]):
            next_depth = depths[i + 1]
            curr_present = pkl_dict[depth][src, tgt] == 1
            next_present = pkl_dict[next_depth][src, tgt] == 1
            
            # Deletion: edge present at depth i, absent at depth i+1
            if curr_present and not next_present:
                deletion_matrix[edge_idx, i] = 1
            # Addition: edge absent at depth i, present at depth i+1
            elif not curr_present and next_present:
                addition_matrix[edge_idx, i] = 1
    
    return deletion_matrix, addition_matrix, edge_labels, depths


# Generate heatmaps for each recording-method combo (always run for visualizations)
if not outputs_exist:
    def load_adjacency_dict(pkl_path):
        """Load adjacency dictionary from pickle file."""
        with open(pkl_path, 'rb') as f:
            adj_dict = pickle.load(f)
        return adj_dict

    for method, recordings in [('c-GC', cgc_recordings), ('c-GC-star', cgc_star_recordings)]:
        print(f"\nGenerating heatmaps for {method}...")
        
        for recording_id, pkl_path in list(sorted(recordings.items()))[:2]:  # Limit to 2 for notebook preview
            try:
                adj_dict = load_adjacency_dict(pkl_path)
                result = create_edge_heatmaps(adj_dict, recording_id, method, output_dir)
                
                if result[0] is not None:
                    deletion_matrix, addition_matrix, edge_labels, depths = result
                    
                    # Plot deletions
                    fig, axes = plt.subplots(1, 2, figsize=(14, min(12, len(edge_labels) * 0.2)))
                    
                    # Deletion heatmap
                    sns.heatmap(deletion_matrix, cmap='Reds', cbar=True, ax=axes[0],
                               xticklabels=[f"{d}→{depths[i+1]}" for i, d in enumerate(depths[:-1])],
                               yticklabels=edge_labels if len(edge_labels) <= 50 else [])
                    axes[0].set_title(f"Edge Deletions - {recording_id} ({method})")
                    axes[0].set_ylabel("Edge")
                    axes[0].set_xlabel("Depth Transition")
                    
                    # Addition heatmap
                    sns.heatmap(addition_matrix, cmap='Blues', cbar=True, ax=axes[1],
                               xticklabels=[f"{d}→{depths[i+1]}" for i, d in enumerate(depths[:-1])],
                               yticklabels=edge_labels if len(edge_labels) <= 50 else [])
                    axes[1].set_title(f"Edge Additions - {recording_id} ({method})")
                    axes[1].set_ylabel("Edge")
                    axes[1].set_xlabel("Depth Transition")
                    
                    plt.tight_layout()
                    
                    # Save figure
                    fig_name = f"{recording_id}_{method}_edge_heatmaps.png"
                    fig_path = output_dir / fig_name
                    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
                    print(f"  Saved: {fig_path.name}")
                    plt.close(fig)
            except Exception as e:
                print(f"  Error processing {recording_id}: {str(e)}")
else:
    print("Skipping heatmap generation (cached outputs)")

## Summary Statistics

In [ ]:
print("\\n" + "="*60)
print("EDGE LOCALIZATION SUMMARY")
print("="*60)

print(f"\\nTotal edges: {len(edge_metrics_df)}")
print(f"\\nEdge status distribution:")
print(edge_metrics_df['status'].value_counts())

print(f"\\nInstability frequency statistics:")
print(edge_metrics_df['instability_frequency'].describe())

print(f"\\nEdges by recording:")
print(edge_metrics_df.groupby('recording').size())

print(f"\\nEdges by method:")
print(edge_metrics_df.groupby('method').size())

print(f"\\nMost unstable edges (by instability frequency):")
top_unstable = edge_metrics_df[edge_metrics_df['status'] == 'unstable'].nlargest(10, 'instability_frequency')
print(top_unstable[['recording', 'method', 'source', 'target', 'instability_frequency']].to_string())

## Create manifest for reproducibility

In [ ]:
logger.info("Creating manifest...")

if not outputs_exist:
    import subprocess

    # Get git commit hash
    try:
        git_commit = subprocess.check_output(
            ['git', 'rev-parse', '--short', 'HEAD'],
            cwd=project_root
        ).decode().strip()
    except:
        git_commit = 'unknown'

    manifest = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'git_commit': git_commit,
        'analysis': 'phase 4.2: edge_localization',
        'input_paths': [
            str(outputs_v2a / 'c-GC'),
            str(outputs_v2a / 'c-GC-star')
        ],
        'output_paths': [
            'edge_instability.csv',
            'edge_fdr.csv' if 'q_value' in edge_metrics_df.columns else None,
        ],
        'method': 'edgewise instability localization',
        'method_params': {
            'edge_tracking': 'per-edge instability frequency',
            'metrics': ['n_depths_present', 'n_deletions', 'n_additions', 
                       'instability_frequency', 'status']
        },
        'recordings_processed': len(cgc_recordings) + len(cgc_star_recordings),
        'total_edges': len(edge_metrics_df),
        'edge_status_counts': {
            'stable': int((edge_metrics_df['status'] == 'stable').sum()),
            'unstable': int((edge_metrics_df['status'] == 'unstable').sum()),
            'novel': int((edge_metrics_df['status'] == 'novel').sum()),
            'lost': int((edge_metrics_df['status'] == 'lost').sum()),
        },
        'software_versions': {
            'python': '3.11.0',
            'numpy': str(np.__version__),
            'pandas': str(pd.__version__),
            'matplotlib': str(plt.matplotlib.__version__),
        }
    }

    # Remove None values
    manifest['output_paths'] = [p for p in manifest['output_paths'] if p]

    # Save manifest
    manifest_path = output_dir / 'manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    logger.info(f"Manifest saved to: {manifest_path}")
    print(f"\nManifest saved to: {manifest_path}")
else:
    logger.info("Skipping manifest creation (loading from cache)")
    print("Skipping manifest creation (loading from cache)")

## Phase 4.2 Complete

✅ Edge-level instability metrics computed for all v2a-RSN recordings
✅ Outputs exported: CSV files with edge metrics
✅ Heatmaps generated showing edge additions and deletions
✅ Manifest created for reproducibility

**Output Directory**: `outputs/v2a-RSNs/edge_localization/`